# Sleep-EDF — финальный inference113 /099

Один зафиксированный notebook, без обучения, из raw test и сохранённых весов.
`OUTPUT_SECOND_SUBMISSION=False`: `submission.csv` (113).
`True`: дополнительно `submission_2.csv` (099), без повторного inference backbones.

Подключите full_psg_v1 version1, `sleep-edf-final-offline-inputs.zip` (код/веса)
и `sleep-edf-final-runtime-linux.zip` (готовые Linux-окружения).
Среда Kaggle: T4×2, Internet Off для всего финального запуска.
Публикация одного notebook для двух решений согласована с преподавателем.

Проверка готовности: полный run должен получить `EXACT_MATCH`,66222 ID и ноль
несовпадающих меток для каждого включённого решения. Эталонные CSV служат только
для сравнения — не подменяют predictions. Этот исходный notebook пока не выполнен на Kaggle.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
import time
from pathlib import Path

NOTEBOOK_STARTED = time.perf_counter()
OUTPUT_SECOND_SUBMISSION = False
DEVICE = "cuda:0"
THREADS = max(1, os.cpu_count() or 2)
FEATURE_WORKERS = min(2, THREADS)
OUTPUT_ROOT = Path("/kaggle/working/sleep-edf-final-offline")
FINAL_ROOT = None  # Автопоиск нового Input bundle; GitHub и токен не нужны.
DATA_ROOT = None  # Автопоиск исходного full_psg_v1 version1.
print(
    "Включён099:", OUTPUT_SECOND_SUBMISSION, "· GPU:", DEVICE, "· CPU threads:", THREADS
)
RUNTIME_INPUT = None  # Автопоиск отдельного offline-runtime Input.
RUNTIME_CACHE = Path("/kaggle/working/_sleep-edf-offline-runtimes")

In [ ]:
import hashlib
from zipfile import ZipFile


def unpack(path):
    key = hashlib.sha256(
        (str(path) + str(path.stat().st_size) + str(path.stat().st_mtime_ns)).encode()
    ).hexdigest()[:16]
    target = Path("/kaggle/working/_final_input") / key
    if (target / ".unpacked").exists():
        return target
    target.mkdir(parents=True, exist_ok=True)
    with ZipFile(path) as z:
        for info in z.infolist():
            dest = (target / info.filename).resolve()
            if (
                not dest.is_relative_to(target.resolve())
                or (info.external_attr >> 16) & 0o170000 == 0o120000
            ):
                raise ValueError("Unsafe ZIP member")
            z.extract(info, target)
    (target / ".unpacked").touch()
    return target


if FINAL_ROOT is None:
    candidates = [
        p.parent
        for p in Path("/kaggle/input").rglob("bundle.json")
        if json.loads(p.read_text()).get("kind") == "sleep_edf_final_v2"
    ]
    if not candidates:
        for archive in Path("/kaggle/input").rglob("*.zip"):
            with ZipFile(archive) as z:
                if (
                    "bundle.json" in z.namelist()
                    and json.loads(z.read("bundle.json")).get("kind")
                    == "sleep_edf_final_v2"
                ):
                    candidates.append(unpack(archive))
    if len(candidates) != 1:
        raise RuntimeError(
            "Нужен новый sleep-edf-final-offline-inputs bundle; прежний Input v1 не подходит"
        )
    FINAL_ROOT = candidates[0] / "final"
BUNDLE_ROOT = FINAL_ROOT.parent
manifest = json.loads((BUNDLE_ROOT / "bundle.json").read_text())
for name, expected in manifest["files"].items():
    path = (BUNDLE_ROOT / name).resolve()
    if not path.is_relative_to(BUNDLE_ROOT.resolve()):
        raise ValueError("Unsafe bundle path")
    with path.open("rb") as f:
        actual = hashlib.file_digest(f, "sha256").hexdigest()
    if actual != expected:
        raise ValueError(f"Input checksum mismatch: {name}")
if DATA_ROOT is None:
    roots = [
        p.parent
        for p in Path("/kaggle/input").rglob("MANIFEST.json")
        if (p.parent / "epochs/test").is_dir()
        and (p.parent / "sample_submission.csv").is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError(f"Нужен один исходный full_psg_v1: {roots}")
    DATA_ROOT = roots[0]
print("Код/веса:", FINAL_ROOT, "\nДанные:", DATA_ROOT)
print("Исходный dataset: levshaazz/sleep-edf-full-psg-course-data, version1")

In [ ]:
import html

from IPython.display import HTML, display

spec = importlib.util.spec_from_file_location(
    "final_notebook_process", FINAL_ROOT / "notebook_process.py"
)
process_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(process_module)
run_command = process_module.run_command
process_module.reset_commands()

runtime_spec = importlib.util.spec_from_file_location(
    "sleep_edf_offline_runtime", FINAL_ROOT / "offline_runtime.py"
)
runtime_module = importlib.util.module_from_spec(runtime_spec)
runtime_spec.loader.exec_module(runtime_module)
if RUNTIME_INPUT is None:
    RUNTIME_INPUT = runtime_module.discover(Path("/kaggle/input"))
print("Offline runtime Input:", RUNTIME_INPUT)
handle = display(HTML("<b>Подготовка offline-окружений</b>"), display_id=True)


def show_runtime_stage(text):
    handle.update(HTML("<b>Подготовка offline-окружений</b><br>" + html.escape(text)))


runtimes, runtime_manifest = runtime_module.prepare(
    RUNTIME_INPUT, RUNTIME_CACHE, FINAL_ROOT, show_runtime_stage
)
FEATURE_PYTHON = runtimes["features"]
BASE = [str(runtimes["inference"]), "-I"]
ENV = {
    "PYTHONHOME": "",
    "PYTHONPATH": "",
    "VIRTUAL_ENV": "",
    "PYTHONNOUSERSITE": "1",
    "UV_OFFLINE": "1",
    "UV_PYTHON_DOWNLOADS": "never",
    "PIP_NO_INDEX": "1",
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
    "MPLBACKEND": "Agg",
    **{
        k: str(THREADS)
        for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")
    },
}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "offline_runtime_manifest.json").write_text(
    json.dumps(runtime_manifest, indent=2) + "\n"
)
subprocess.run(["nvidia-smi"], check=True)
print("Notebook kernel:", sys.version)
print(
    "Восстановлены Python3.12.12 и3.14.2 с пакетами из Input. Скачиваний и pip/uv install нет."
)

In [ ]:
command = [
    *BASE,
    str(FINAL_ROOT / "run.py"),
    "--data-root",
    str(DATA_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
    "--feature-python",
    str(FEATURE_PYTHON),
    "--device",
    DEVICE,
    "--threads",
    str(THREADS),
    "--feature-workers",
    str(FEATURE_WORKERS),
]
if OUTPUT_SECOND_SUBMISSION:
    command.append("--output-second-submission")
run_command(command, cwd=FINAL_ROOT, env=ENV)
report = json.loads((OUTPUT_ROOT / "run_report.json").read_text())
assert report["status"] == "EXACT_MATCH" and report["epochs"] == 66222
assert all(item["label_mismatches"] == 0 for item in report["comparisons"].values())
print("Проверено полное совпадение:", report["comparisons"])

Последнюю ячейку можно выполнить отдельно после ошибки: она сохранит диагностические
результаты, если run_report.json уже создан. В финальную сдачу нужен только успешный
полный запуск с сохранёнными outputs. После него сохраните точную Kaggle Version,
скачайте выполненный.ipynb и его PDF; добавьте version и submission IDs в метаданные.


In [ ]:
from datetime import UTC, datetime

from IPython.display import FileLink, display

elapsed = time.perf_counter() - NOTEBOOK_STARTED
print(f"Время всех ячеек, включая установку окружений: {elapsed:.1f}s")
if (OUTPUT_ROOT / "run_report.json").exists():
    report = json.loads((OUTPUT_ROOT / "run_report.json").read_text())
    report["notebook_elapsed_seconds"] = elapsed
    report["within_two_hours_including_setup"] = elapsed <= 7200
    (OUTPUT_ROOT / "run_report.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2) + "\n"
    )
    (OUTPUT_ROOT / "input_bundle_manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n"
    )
    stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
    archive = Path("/kaggle/working") / f"sleep-edf-final-results-{stamp}.zip"
    run_command(
        [
            *BASE,
            str(FINAL_ROOT / "package.py"),
            "results",
            "--results-root",
            str(OUTPUT_ROOT),
            "--output",
            str(archive),
        ],
        cwd=FINAL_ROOT,
        env=ENV,
    )
    for filename in (
        ["submission.csv", "submission_2.csv"]
        if OUTPUT_SECOND_SUBMISSION
        else ["submission.csv"]
    ):
        path = OUTPUT_ROOT / filename
        if path.exists():
            display(
                FileLink(
                    str(path.relative_to(Path.cwd())),
                    result_html_prefix=filename + ": ",
                )
            )
    display(
        FileLink(
            str(archive.relative_to(Path.cwd())),
            result_html_prefix="Все результаты одним ZIP: ",
        )
    )
    if report["status"] == "EXACT_MATCH":
        assert elapsed <= 7200, (
            "Лимит2часа превышен; run пока нельзя считать готовым финалом"
        )